In [ ]:
#| default_exp eval

In [ ]:
#| export
from __future__ import annotations

import math
from collections.abc import Callable
from dataclasses import asdict, dataclass

import numpy as np
import torch
import torch.nn as nn

In [ ]:
#| include: false
from nbdev.showdoc import *

## Overview

An accuracy published alone is not comparable to anything. This harness keeps the **per-image** result, so
two models evaluated on the same images can be compared as a paired sample instead of two lonely numbers.

| Function | Answers |
|---|---|
| `correct_vector` | which images this model gets right |
| `wilson` | how wide the interval around `k/n` is |
| `paired_delta` | is the difference with the reference distinguishable from zero |
| `agreement` | do two artifacts of the same model predict the same class |

A NaN logit or an empty dataloader raises: a silent 0 % is worse than a stack trace.

In [ ]:
#| export
def _run(model, dl, device):
    "One pass over `dl`: predicted classes and targets"
    if isinstance(model, nn.Module): model = model.to(device).eval()
    preds, targets = [], []
    with torch.no_grad():
        for x, y in dl:
            out = model(x.to(device) if torch.is_tensor(x) else x)
            out = out if torch.is_tensor(out) else torch.as_tensor(np.asarray(out))
            if not torch.isfinite(out).all():
                raise ValueError("model returned logits that are not finite — evaluate a finite model")
            preds.append(out.detach().cpu().argmax(-1).numpy().reshape(-1))
            targets.append(np.asarray(y.detach().cpu() if torch.is_tensor(y) else y).reshape(-1))
    if not preds: raise ValueError("dataloader is empty — nothing to evaluate")
    return np.concatenate(preds), np.concatenate(targets)


def predictions(
    model: Callable,                     # a model, or anything callable on a batch of inputs
    dl,                                  # dataloader yielding (inputs, targets)
    device: str | torch.device = 'cpu',  # device the inputs are moved to
) -> np.ndarray:
    "Predicted class of every image, in dataloader order"
    return _run(model, dl, device)[0]


def correct_vector(
    model: Callable,                     # a model, or anything callable on a batch of inputs
    dl,                                  # dataloader yielding (inputs, targets)
    device: str | torch.device = 'cpu',  # device the inputs are moved to
) -> np.ndarray:
    "Per-image correctness, in dataloader order"
    preds, targets = _run(model, dl, device)
    return preds == targets

In [ ]:
show_doc(predictions)

In [ ]:
show_doc(correct_vector)

In [ ]:
#| export
def wilson(
    k: int,           # images classified correctly
    n: int,           # images evaluated
    z: float = 1.96,  # 1.96 for a 95 % interval
) -> tuple[float, float]:
    "Wilson score interval of an accuracy, as fractions"
    if n <= 0: raise ValueError("wilson needs n > 0 — pass the number of images evaluated")
    p, d = k / n, 1 + z * z / n
    centre = (p + z * z / (2 * n)) / d
    half = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return max(0., centre - half), min(1., centre + half)

In [ ]:
show_doc(wilson)

In [ ]:
#| export
@dataclass(slots=True)
class PairedDelta:
    "Accuracy difference between two models evaluated on the same images, in percentage points"
    delta: float
    lo: float
    hi: float
    p_mcnemar: float
    n: int

    def as_dict(self) -> dict: return asdict(self)


def _mcnemar(n01, n10):
    "Exact two-sided McNemar p from the discordant counts"
    n = n01 + n10
    if n == 0: return 1.0
    return min(1., 2 * sum(math.comb(n, i) for i in range(min(n01, n10) + 1)) / 2 ** n)


def paired_delta(
    a: np.ndarray,       # per-image correctness of the reference
    b: np.ndarray,       # per-image correctness of the model compared to it
    n_boot: int = 2000,  # bootstrap resamples over the images
    seed: int = 0,       # seed of the resampling
) -> PairedDelta:
    "Accuracy difference b - a in points, with a paired bootstrap interval and the exact McNemar p"
    a, b = np.asarray(a, dtype=bool), np.asarray(b, dtype=bool)
    if a.shape != b.shape: raise ValueError(f"paired vectors must cover the same images, got {a.shape} and {b.shape}")
    n = a.size
    if n == 0: raise ValueError("paired vectors are empty — nothing to compare")
    idx = np.random.default_rng(seed).integers(0, n, size=(n_boot, n))
    boot = 100 * (b[idx].mean(1) - a[idx].mean(1))
    lo, hi = np.percentile(boot, [2.5, 97.5])
    return PairedDelta(float(100 * (b.mean() - a.mean())), float(lo), float(hi),
                       _mcnemar(int((a & ~b).sum()), int((~a & b).sum())), n)

In [ ]:
show_doc(paired_delta)

In [ ]:
show_doc(PairedDelta)

In [ ]:
#| export
def agreement(
    pred_a: np.ndarray,  # predicted classes of one artifact
    pred_b: np.ndarray,  # predicted classes of the other
) -> float:
    "Fraction of images on which two artifacts predict the same class"
    a, b = np.asarray(pred_a), np.asarray(pred_b)
    if a.shape != b.shape: raise ValueError(f"prediction vectors must cover the same images, got {a.shape} and {b.shape}")
    if a.size == 0: raise ValueError("prediction vectors are empty — nothing to compare")
    return float((a == b).mean())

In [ ]:
show_doc(agreement)

---

## Usage

```python
from fastermodels import correct_vector, wilson, paired_delta, predictions, agreement

ref = correct_vector(source_model, valid_dl)
opt = correct_vector(FasterModel.from_pretrained('artifact'), valid_dl)

k, n = int(opt.sum()), opt.size
wilson(k, n)                 # (0.9298, 0.9448) — the interval that belongs next to k/n
paired_delta(ref, opt)       # PairedDelta(delta=-0.51, lo=-1.2, hi=0.2, p_mcnemar=0.12, n=3925)
agreement(predictions(source_model, valid_dl), predictions(reloaded, valid_dl))
```

`paired_delta` compares the two vectors image by image, so it reads a small difference the two intervals
would leave undecided. `lo > floor` is the publication condition; `delta` alone is not.

---

## See Also

- [Model](00_model.html) - the model these numbers are measured on
- [Card](02_card.html) - where `k`, `n`, the interval and the delta are written down
- [Gate](03_gate.html) - the condition that reads `lo` against the floor

Tests live in `nbs/tests/test_eval.ipynb`.